# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [0 1 1]` — `/` это обычное деление и даёт float; `//` целочисленное деление: 2//4=0, 4//4=1, 6//4=1.
2. `[4. 4. 4.]` — `axis=1` суммирует по столбцам внутри каждой строки; в каждой строке 4 единицы, строк 3.
3. `[[0 1 2] [1 2 3] [2 3 4]]` — broadcasting: строка (3,) и столбец (3, 1) растягиваются до (3, 3), элемент [i, j] = i + j.
4. `6.0` — `sqrt` даёт [1, 2, 3], сумма равна 6.0.
5. `7.5` — квадраты 1, 4, 9, 16, их среднее (1+4+9+16)/4 = 7.5.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1
months = np.unique(month)
print("Месяцы в данных:", months)
m1, m2 = months[0], months[1]          # сентябрь и октябрь

print("Строк в clean:", clean.sum(), "из", len(oid))
rev = np.array([amount[clean & (month == m)].sum() for m in (m1, m2)])
chg = (rev[1] - rev[0]) / rev[0] * 100
print(f"Выручка {m1}: {rev[0]:,.0f} ₸ | {m2}: {rev[1]:,.0f} ₸ | изменение: {chg:+.1f}%")

# для сравнения с ДЗ 2: что было бы без чистки
raw = np.array([amount[paid & (month == m)].sum() for m in (m1, m2)])
raw_chg = (raw[1] - raw[0]) / raw[0] * 100
print(f"Без чистки (оплаченные, с дублями и TEST): {raw[0]:,.0f} ₸ -> {raw[1]:,.0f} ₸ ({raw_chg:+.1f}%)")
print("Сравните эти числа с вашим ответом в ДЗ 2: расхождение даёт чистка (дубли и TEST).")
print(f"Менеджеру: по чистым данным выручка {'упала' if chg < 0 else 'выросла'} на {abs(chg):.1f}% "
      f"({rev[0]:,.0f} -> {rev[1]:,.0f} ₸).")

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2
cats = np.unique(cat)
print("Категории:", cats, "| штук:", len(cats))
revenue = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in (m1, m2)] for c in cats])
print("Форма таблицы:", revenue.shape)

print(f"{'Категория':<15}{m1:>14}{m2:>14}{'Всего':>14}{'Изм., %':>10}")
total_cat = revenue.sum(axis=1)                       # сумма по месяцам
chg_cat = (revenue[:, 1] - revenue[:, 0]) / revenue[:, 0] * 100
for k, c in enumerate(cats):
    print(f"{c:<15}{revenue[k,0]:>14,.0f}{revenue[k,1]:>14,.0f}{total_cat[k]:>14,.0f}{chg_cat[k]:>+10.1f}")

top, worst = total_cat.argmax(), chg_cat.argmin()
print(f"Менеджеру: больше всех за два месяца принесла «{cats[top]}» ({total_cat[top]:,.0f} ₸), "
      f"сильнее всех упала «{cats[worst]}» ({chg_cat[worst]:+.1f}%).")

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3
counts = np.array([[(clean & (cat == c) & (month == m)).sum() for m in (m1, m2)] for c in cats])
avg_check = np.divide(revenue, counts, out=np.zeros_like(revenue), where=counts > 0)   # одна операция над двумя таблицами

print(f"{'Категория':<15}{'заказов '+m1:>16}{'заказов '+m2:>16}{'чек '+m1:>14}{'чек '+m2:>14}")
for k, c in enumerate(cats):
    print(f"{c:<15}{counts[k,0]:>16}{counts[k,1]:>16}{avg_check[k,0]:>14,.0f}{avg_check[k,1]:>14,.0f}")

best = avg_check[:, 1].argmax()
print(f"Менеджеру: в {m2} самый высокий средний чек у «{cats[best]}» — {avg_check[best,1]:,.0f} ₸ "
      f"({counts[best,1]} заказов).")

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in (m1, m2)] for c in cities])
share = table / table.sum(axis=0)                      # доля города в выручке своего месяца

print("Сумма долей по столбцам:", share.sum(axis=0), "| равна 1:", np.allclose(share.sum(axis=0), 1))
print(f"{'Город':<12}{m1:>10}{m2:>10}{'Δ, п.п.':>10}")
for k, c in enumerate(cities):
    print(f"{c:<12}{share[k,0]:>10.1%}{share[k,1]:>10.1%}{(share[k,1]-share[k,0])*100:>+10.1f}")

s = list(cities).index("Шымкент")
print(f"Менеджеру: доля Шымкента изменилась с {share[s,0]:.1%} до {share[s,1]:.1%} "
      f"({(share[s,1]-share[s,0])*100:+.1f} п.п.).")

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5
delta = table[:, 1] - table[:, 0]                      # изменение выручки по городам
total_delta = delta.sum()
contrib = delta / total_delta * 100                    # вклад города в общее изменение, %

print(f"Общее изменение выручки: {total_delta:+,.0f} ₸")
for k, c in enumerate(cities):
    print(f"  {c:<12}{delta[k]:>+14,.0f} ₸   вклад {contrib[k]:>+7.1f}%")
print("Сумма вкладов:", round(contrib.sum(), 6), "%")

s = list(cities).index("Шымкент")
main = delta.argmin() if total_delta < 0 else delta.argmax()
print(f"Шымкент даёт {contrib[s]:.1f}% общего {'падения' if total_delta < 0 else 'изменения'}.")
print(f"Менеджеру: больше всего на динамику повлиял город {cities[main]} ({delta[main]:+,.0f} ₸, "
      f"{contrib[main]:.1f}% общего изменения) — причину нужно искать в первую очередь там.")

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6
print("table.shape:", table.shape, "| table.sum(axis=1).shape:", table.sum(axis=1).shape)
try:
    share_bad = table / table.sum(axis=1)
except ValueError as e:
    print("Ошибка:", e)

# Почему падает: table.sum(axis=1) имеет форму (число_городов,), и broadcasting
# выравнивает формы справа: (n, 2) и (n,) -> сравниваются 2 и n, они не совпадают.
# Сумма по строкам должна быть столбцом (n, 1): для этого keepdims=True.
share_city = table / table.sum(axis=1, keepdims=True)
print("Сумма по строкам:", share_city.sum(axis=1), "| все равны 1:", np.allclose(share_city.sum(axis=1), 1))
print(f"{'Город':<12}{m1:>10}{m2:>10}")
for k, c in enumerate(cities):
    print(f"{c:<12}{share_city[k,0]:>10.1%}{share_city[k,1]:>10.1%}")
print("Менеджеру: исправленный код считает долю каждого месяца внутри города; в сумме по городу выходит 100%.")

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
out = np.abs(z) > 3
print(f"Выбросов |z| > 3: {out.sum()} из {len(x)} ({out.mean():.2%})")
print(f"Среднее z: {z.mean():.2e} (≈ 0: {np.isclose(z.mean(), 0)}) | std z: {z.std():.4f} (= 1: {np.isclose(z.std(), 1)})")
if out.any():
    print("Суммы выбросов, ₸:", np.sort(x[out])[::-1][:10])
print(f"Менеджеру: необычно крупных заказов (больше 3 стандартных отклонений) {out.sum()} — {out.mean():.1%} от всех; "
      ("их стоит проверить вручную." if out.any() else "аномально крупных заказов нет."))

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8
ok = clean & ~np.isnan(days)
X = np.column_stack([amount[ok], days[ok]])
print("Форма X:", X.shape)
print("До масштабирования: min =", X.min(axis=0), "| max =", X.max(axis=0))

X01 = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print("После: min =", X01.min(axis=0), "| max =", X01.max(axis=0))
print("Менеджеру: сумма и срок доставки приведены к одной шкале [0, 1], теперь модель сможет честно сравнивать их влияние.")

**Зачем признаки в одном масштабе.** Сумма заказа измеряется десятками тысяч, срок доставки — единицами дней. Модели, которые считают расстояния или двигают веса градиентным спуском, увидят в сумме «главный» признак просто из-за больших чисел, а доставку почти проигнорируют. После приведения к [0, 1] оба признака вносят сопоставимый вклад, и модель различает их по смыслу, а не по единицам измерения.